<a href="https://colab.research.google.com/github/vthkrl/big_data_fall26/blob/main/Big_Data_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Data cleaning: standardize date ranges and spatial units from each data source, handle facilities with poor or absent location information.


See `blockify_years.ipynb` for various cleaning tasks - notably, converting yearly data into five-year blocks for the crime rate, fine particles, median income, and premature mortality metrics. This notebook operates on the cleaned CSVs produced by that one.

All the metric tables consumed by this notebook should have the following columns:

- `TimePeriod`, holding the five-year block (in the format `YYYY-YY`, ie `2015-19`)
- `GeoID`, holding the community district ID number
- Some other column with the actual metric value for that time period and community district

The name of the metric-value column should be specified in the `metric_colnames` table, along with the name to be used for that metric in the merged table.

<br>

**TODO: does the facilities table need cleanup?**



2. Data consolidation: combine information from a wide range of data sources to build a comprehensive picture of each NYC community district.




Load and combine data from cleaned CSVs

In [9]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')
folder_path = "/content/drive/MyDrive/Project/datasets/"

spark = SparkSession.builder \
    .appName("FacilityPredictor") \
    .getOrCreate()

facilities_path = folder_path + "facilities.csv"
facilities = spark.read.csv(facilities_path, header=True, inferSchema=True)

# metrics
# 1. rent burden
rent_burden_rate_path = folder_path + "rent_burden.csv"
rent_burden_rate = spark.read.csv(rent_burden_rate_path, header=True, inferSchema=True)

# 2. crowding
crowding_rate_path = folder_path + "crowding.csv"
crowding_rate = spark.read.csv(crowding_rate_path, header=True, inferSchema=True)

# 3. median income
median_income_path = folder_path + "median_income_parsed.csv"
median_income = spark.read.csv(median_income_path, header=True, inferSchema=True)

# 4. poverty
poverty_rate_path = folder_path + "poverty.csv"
poverty_rate = spark.read.csv(poverty_rate_path, header=True, inferSchema=True)

# 5. crime
crime_rate_path = folder_path + "crime_rate_parsed.csv"
crime_rate = spark.read.csv(crime_rate_path, header=True, inferSchema=True)

# 6. high school graduation
high_school_graduation_rate_path = folder_path + "high_school.csv"
high_school_graduation_rate = spark.read.csv(high_school_graduation_rate_path, header=True, inferSchema=True)

# 7. fine particles
fine_particles_concentration_path = folder_path + "fine_particles_parsed.csv"
fine_particles_concentration = spark.read.csv(fine_particles_concentration_path, header=True, inferSchema=True)

# 8. walking to work rate
walk_to_work_rate = folder_path + "walk_to_work.csv"
walk_to_work_rate = spark.read.csv(walk_to_work_rate, header=True, inferSchema=True)

# 9. premature mortality
premature_mortality_rate_path = folder_path + "mortality_parsed.csv"
premature_mortality_rate = spark.read.csv(premature_mortality_rate_path, header=True, inferSchema=True)

# 10. independency
independency_rate_path = folder_path + "indep_living.csv"
independency_rate = spark.read.csv(independency_rate_path, header=True, inferSchema=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
metric_colnames = [
    (rent_burden_rate, "Percent", "rent_burden_pct"),
    (crowding_rate, "Percent", "crowding_pct"),
    (median_income, "median_income", "median_income"),
    (poverty_rate, "Percent", "poverty_pct"),
    (crime_rate, "crime_all_rt", "crimes_per_1000_ppl"),
    (high_school_graduation_rate, "Percent", "high_school_grad_pct"),
    (fine_particles_concentration, "Annual mean mcg/m3", "fine_particle_concentration"),
    (walk_to_work_rate, "Walked (percent)", "walk_to_work_rate"),
    (premature_mortality_rate, "Age-adjusted rate per 100,000", "early_deaths_per_100000_ppl"),
    (independency_rate, "Percent", "independency_pct")
]

# join all the individual metric tables by TimePeriod and GeoID, using left join to handle time periods absent from some datasets
mega_join = rent_burden_rate
select_terms = ["TimePeriod", "GeoID"]
for group in metric_colnames:
  if group[0] != rent_burden_rate:
    mega_join = mega_join.join(group[0], ["TimePeriod", "GeoID"], how="left")
  # create the appropriate term for the select statement, ie 'crowding_rate["Percent"].alias("crowding_pct")'
  select_terms.append(group[0][group[1]].alias(group[2]))

# select only the columns we actually care about
all_metrics = mega_join.select(*select_terms)
all_metrics.show(10)

+----------+-----+---------------+------------+-------------+-----------+-------------------+--------------------+---------------------------+-----------------+---------------------------+----------------+
|TimePeriod|GeoID|rent_burden_pct|crowding_pct|median_income|poverty_pct|crimes_per_1000_ppl|high_school_grad_pct|fine_particle_concentration|walk_to_work_rate|early_deaths_per_100000_ppl|independency_pct|
+----------+-----+---------------+------------+-------------+-----------+-------------------+--------------------+---------------------------+-----------------+---------------------------+----------------+
|   2015-19|  201|           58.4|        12.7|      30722.2|       40.1|              24.98|                63.8|                       8.54|              9.7|                     310.22|             8.5|
|   2015-19|  202|           58.4|        12.7|      30722.2|       40.1|              19.32|                63.8|                       8.54|              9.7|                

In [20]:
# drop rows with any NULL values (or comment out this code) *****
all_metrics = all_metrics.dropna()

Calculate domain scores and percentiles

In [12]:
# community districts
geoids = [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 501, 502, 503]
geoids_tuples = [(geoid,) for geoid in geoids]
scores_start = spark.createDataFrame(geoids_tuples, schema="GeoID int")

# scores
scores = scores_start.join(
    all_metrics,
    on="GeoID",
    how="left"
)

# percentiles
from pyspark.sql import Window

percentiles = scores

for group in metric_colnames:
    col_name = group[2]
    window = Window.partitionBy("TimePeriod").orderBy(F.col(col_name))

    percentiles = percentiles.withColumn(col_name + " percentile", F.percent_rank().over(window))

percentiles.filter(F.col("TimePeriod")=="2015-19").show(10)

+-----+----------+---------------+------------+-------------+-----------+-------------------+--------------------+---------------------------+-----------------+---------------------------+----------------+--------------------------+-----------------------+------------------------+----------------------+------------------------------+-------------------------------+--------------------------------------+----------------------------+--------------------------------------+---------------------------+
|GeoID|TimePeriod|rent_burden_pct|crowding_pct|median_income|poverty_pct|crimes_per_1000_ppl|high_school_grad_pct|fine_particle_concentration|walk_to_work_rate|early_deaths_per_100000_ppl|independency_pct|rent_burden_pct percentile|crowding_pct percentile|median_income percentile|poverty_pct percentile|crimes_per_1000_ppl percentile|high_school_grad_pct percentile|fine_particle_concentration percentile|walk_to_work_rate percentile|early_deaths_per_100000_ppl percentile|independency_pct percen

Calculate overall quality metric

In [17]:
weights = [ # weights for:
    -1,   # rent_burden_pct
    -1,   # crowding_pct
    5,    # median_income
    -2,   # poverty_pct
    -1,   # crimes_per_1000_ppl
    4,    # high_school_grad_pct
    -0.5, # fine_particle_concentration
    3,    # walk_to_work_rate
    -1,   # early_deaths_per_100000_ppl
    2     # independency_pct
]
total_magnitude = sum(abs(w) for w in weights)

norm_weights = [w / total_magnitude for w in weights]

# OverallQuality ranges from 0.0 to 1.0
quality_score = F.lit(0.0)
for i, w in enumerate(norm_weights):
    column = F.col(metric_colnames[i][2] + " percentile")

    norm_metric = (1.0 - column) if w < 0 else column
    quality_score = quality_score + (abs(w) * norm_metric)

overall_quality = percentiles.withColumn("OverallQuality", quality_score)

# comment out this for loop if you want to keep the scores and percentiles
for group in metric_colnames:
  overall_quality = overall_quality.drop(group[2]).drop(group[2] + " percentile")


top_10_cds = overall_quality.orderBy(F.col("OverallQuality").desc()).limit(10)
top_10_cds.show()

+-----+----------+------------------+
|GeoID|TimePeriod|    OverallQuality|
+-----+----------+------------------+
|  108|   2014-18|0.8351555929352397|
|  106|   2014-18|0.8221194280908326|
|  108|   2015-19|0.8221194280908326|
|  106|   2015-19|  0.81959629941127|
|  101|   2014-18|0.8019343986543312|
|  101|   2015-19|0.7994112699747686|
|  102|   2014-18|0.7956265769554247|
|  107|   2014-18| 0.795206055508831|
|  102|   2015-19|0.7935239697224559|
|  107|   2015-19| 0.789739276703112|
+-----+----------+------------------+



Add facility counts columns to make training table

In [18]:
# facilities
facility_counts = (
    facilities
    .groupBy("cd")
    .pivot("facgroup")
    .count()
    .na.fill(0)
).withColumnRenamed("cd", "GeoID")

trainer = overall_quality.join(
    facility_counts,
    on="GeoID",
    how="left"
)

trainer.show(10)

+-----+----------+-------------------+--------------+-----+--------------------------+---------------------------------------------+---------------------+-----------------------------+------------------+-----------+----------------+----------------+--------------+-----------------------+---------+-----------------------------+------------------------------+--------------+----------------+-------------+--------------+-----------+------------------+--------------+----------------------------------+--------------------+--------------+
|GeoID|TimePeriod|     OverallQuality|ADULT SERVICES|CAMPS|CHILD SERVICES AND WELFARE|CITY AGENCY PARKING, MAINTENANCE, AND STORAGE|CULTURAL INSTITUTIONS|DAY CARE AND PRE-KINDERGARTEN|EMERGENCY SERVICES|HEALTH CARE|HIGHER EDUCATION|HISTORICAL SITES|HUMAN SERVICES|JUSTICE AND CORRECTIONS|LIBRARIES|MATERIAL SUPPLIES AND MARKETS|OFFICES, TRAINING, AND TESTING|OTHER PROPERTY|PARKS AND PLAZAS|PUBLIC SAFETY|SCHOOLS (K-12)|SOLID WASTE|TELECOMMUNICATIONS|TRANSPORTAT

In [19]:
trainer.toPandas().to_csv(folder_path + "trainer.csv", index=False)

3. Model training: train a machine learning model on the raw scores for the quality of life domains of each district, so that the model determines how the count of a facility type in a district may affect the raw scores of domains for that district.


4. Model execution: for each (district, facility type) pair, use the model to calculate the predicted domain scores, then calculate the predicted domain percentiles and the Predicted Overall Percentile Lift.


In [ ]:
import re

cds = [row[0] for row in facilities.select("cd").distinct().collect()]
facs = [row[0] for row in facilities.select("facgroup").distinct().collect()]

resultArray = []
for cd in cds:
  counts = facility_counts.filter(f"cd = {cd}").collect()[0]
  # TODO: pass unmodified data into model, predict base score
  baseScore = 0.0
  newRowData = [cd]
  #print(f"processing cd #{cd}")
  for factype in facs:
    modified = counts.asDict()
    modified[factype] += 1
    # TODO: pass data into model, get a predicted score, subtract base score
    predictedLift = 0.0 - baseScore
    newRowData.append(predictedLift)
  resultArray.append(newRowData)

outSchema = "cd int"
for factype in facs:
  outSchema += f", `{factype}` double"
results = spark.createDataFrame(resultArray, outSchema)

results.show()

+---+---------+------------------------------+----------------+------------------+-----------------------------+-----+------------------+-----------------------+--------------+-----------+--------------+--------------+----------------------------------+---------------------+--------------+-----------+--------------------+--------------+----------------+-------------+--------------------------+-----------------------------+---------------------------------------------+----------------+--------------+
| cd|LIBRARIES|OFFICES, TRAINING, AND TESTING|HISTORICAL SITES|EMERGENCY SERVICES|MATERIAL SUPPLIES AND MARKETS|CAMPS|TELECOMMUNICATIONS|JUSTICE AND CORRECTIONS|TRANSPORTATION|SOLID WASTE|HUMAN SERVICES|OTHER PROPERTY|VOCATIONAL AND PROPRIETARY SCHOOLS|CULTURAL INSTITUTIONS|SCHOOLS (K-12)|HEALTH CARE|WATER AND WASTEWATER|YOUTH SERVICES|PARKS AND PLAZAS|PUBLIC SAFETY|CHILD SERVICES AND WELFARE|DAY CARE AND PRE-KINDERGARTEN|CITY AGENCY PARKING, MAINTENANCE, AND STORAGE|HIGHER EDUCATION|ADULT 

5. Civil planning: for each district, compare the Predicted Overall Percentile Lift of every facility type to determine which facility type would most improve the quality of life for that district, and for the city as a whole, compare the highest Predicted Overall Percentile Lift from each district to determine which facility types should be built and in which districts they should be built.


6. Interactive output: display both the input data and our calculated results on a user-friendly interactive map to aid in interpretation.